In [ ]:
# ============================================================
# 1. IMPORTS & GLOBAL CONFIGURATION (COLAB & LOCAL COMPATIBLE)
# ============================================================
import os
import sys
import glob
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix
)

# Optional Google Drive Mount (when executing in Google Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_AVAILABLE = True
except Exception:
    DRIVE_AVAILABLE = False

# Robust dual-environment path resolution:
# Priority 1: Google Drive SentryICU directory (if mounted)
# Priority 2: Local repository directory containing 'src' or 'models'
if os.path.exists('/content/drive/MyDrive/SentryICU'):
    PROJECT_DIR = '/content/drive/MyDrive/SentryICU'
elif os.path.exists(os.path.join(os.getcwd(), 'src')):
    PROJECT_DIR = os.getcwd()
else:
    PROJECT_DIR = '/content/drive/MyDrive/SentryICU'

# Search hierarchy for raw PhysioNet data
DATA_SEARCH_PATHS = [
    os.path.join(PROJECT_DIR, 'data', 'raw', 'training_setA', 'training'),
    os.path.join(PROJECT_DIR, 'data', 'raw', 'training_setA'),
    os.path.join(PROJECT_DIR, 'data', 'raw'),
    os.path.join(os.getcwd(), 'data', 'raw', 'training_setA', 'training'),
    os.path.join(os.getcwd(), 'data', 'raw')
]

DATA_PATH = next((p for p in DATA_SEARCH_PATHS if os.path.exists(p) and (glob.glob(os.path.join(p, '*.psv')) or glob.glob(os.path.join(p, '*.csv')))), DATA_SEARCH_PATHS[0])

RUN_ABLATIONS = True            # Set to True to execute full Module 3 ablation suite
RANDOM_SEED = 42
EPOCHS = 15                     # Shared training budget for both baselines and ablations

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[+] Active Execution Device: {DEVICE}")
print(f"[+] Resolved PROJECT_DIR: {PROJECT_DIR}")
print(f"[+] Targeted DATA_PATH:  {DATA_PATH}")

In [ ]:
# ============================================================
# 2. DATASET LOADING & SYNTHETIC FALLBACK GENERATOR
# ============================================================
VITAL_COLS = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp']
LAB_COLS   = ['Lactate', 'WBC', 'Creatinine', 'BUN', 'Glucose', 'Platelets', 'Hgb']

def generate_synthetic_cohort(num_patients=120, max_hours=36):
    """Generates synthetic patient PSV-style data for fallback Colab execution."""
    patient_records = []
    print("[*] Local dataset path not found. Generating synthetic ICU cohort for demonstration...")

    for pid in range(num_patients):
        num_hours = random.randint(18, max_hours)
        is_septic = 1 if random.random() < 0.2 else 0
        sepsis_onset = random.randint(12, num_hours - 1) if is_septic else 999

        base_hr = random.uniform(60, 90) + (15 if is_septic else 0)
        base_temp = random.uniform(36.5, 37.5) + (1.2 if is_septic else 0)
        age = random.uniform(40, 85)
        gender = random.choice([0.0, 1.0])

        for h in range(num_hours):
            sepsis_flag = 1 if h >= sepsis_onset else 0
            row = {
                'PatientID': f'p{pid:06d}',
                'Hour': h,
                'HR': base_hr + random.gauss(0, 3) + (h * 0.5 if sepsis_flag else 0),
                'O2Sat': max(85.0, 98.0 - random.gauss(0, 1) - (h * 0.2 if sepsis_flag else 0)),
                'Temp': base_temp + random.gauss(0, 0.2),
                'SBP': 120.0 + random.gauss(0, 8) - (h * 0.8 if sepsis_flag else 0),
                'MAP': 85.0 + random.gauss(0, 5) - (h * 0.5 if sepsis_flag else 0),
                'DBP': 70.0 + random.gauss(0, 5),
                'Resp': 16.0 + random.gauss(0, 2) + (h * 0.3 if sepsis_flag else 0),
                'Lactate': 1.0 + (random.uniform(0.5, 3.0) if sepsis_flag else 0.1),
                'WBC': 7.0 + (random.uniform(2.0, 8.0) if sepsis_flag else 0.5),
                'Creatinine': 0.9 + random.uniform(0.1, 0.5),
                'BUN': 14.0 + random.uniform(1.0, 4.0),
                'Glucose': 110.0 + random.gauss(0, 10),
                'Platelets': 220.0 - random.uniform(0, 30),
                'Hgb': 12.5 + random.gauss(0, 0.5),
                'Age': age,
                'Gender': gender,
                'ICULOS': h + 1,
                'SepsisLabel': sepsis_flag
            }
            patient_records.append(row)

    df = pd.DataFrame(patient_records)
    return df

def load_icu_data(data_path):
    """Loads dataset from PSV/CSV files or falls back to synthetic generation."""
    if os.path.exists(data_path):
        psv_files = glob.glob(os.path.join(data_path, "*.psv")) + glob.glob(os.path.join(data_path, "*", "*.psv"))
        if psv_files:
            print(f"[+] Found {len(psv_files)} raw PSV patient files in {data_path}.")
            records = []
            for filepath in psv_files[:200]:  # Cap for quick Colab demonstration
                pid = os.path.basename(filepath).replace('.psv', '')
                pdf = pd.read_csv(filepath, sep='|')
                pdf['PatientID'] = pid
                if 'Hour' not in pdf.columns:
                    pdf['Hour'] = np.arange(len(pdf))
                records.append(pdf)
            return pd.concat(records, ignore_index=True)

        csv_files = glob.glob(os.path.join(data_path, "*.csv"))
        if csv_files:
            print(f"[+] Loading dataset from CSV: {csv_files[0]}")
            return pd.read_csv(csv_files[0])

    return generate_synthetic_cohort()

raw_df = load_icu_data(DATA_PATH)

REQUIRED_COLUMNS = set(VITAL_COLS + LAB_COLS + ['PatientID', 'Hour', 'SepsisLabel', 'Age', 'Gender', 'ICULOS'])
missing_cols = REQUIRED_COLUMNS - set(raw_df.columns)
if missing_cols:
    raise ValueError(
        f"Loaded dataset is missing required columns: {sorted(missing_cols)}.\n"
        f"Expected schema: {sorted(REQUIRED_COLUMNS)}.\n"
        "Check DATA_PATH / file format before proceeding."
    )

n_positive = int(raw_df['SepsisLabel'].sum())
if n_positive == 0:
    raise ValueError(
        "Loaded dataset has zero positive SepsisLabel rows. "
        "Check DATA_PATH and source files."
    )

print(f"[+] Schema validated: {n_positive} positive-label rows out of {len(raw_df)} total rows.")

In [ ]:
# ============================================================
# 3. PATIENT-LEVEL STRATIFIED SPLIT & LEAK-FREE PREPROCESSING
# ============================================================
# Extract patient labels for stratification to prevent validation class collapse
patient_labels_series = raw_df.groupby('PatientID')['SepsisLabel'].max()
patient_ids = patient_labels_series.index.values
patient_labels = patient_labels_series.values

# Stratified patient-level split (Prevents both patient data leakage and label imbalance)
train_pids, val_pids = train_test_split(
    patient_ids, test_size=0.2, random_state=RANDOM_SEED, stratify=patient_labels
)

print(f"[+] Stratified Split: {len(train_pids)} Train Patients, {len(val_pids)} Val Patients")
print(f"[+] Train Sepsis Rate: {patient_labels_series.loc[train_pids].mean():.3f} | Val Sepsis Rate: {patient_labels_series.loc[val_pids].mean():.3f}")

def compute_73d_summary(window_df):
    """Computes canonical 73-D static summary vector matching Module 1 / 2 schema."""
    features = []
    # 14 continuous variables (7 vitals + 7 labs) x 5 stats = 70 features
    for col in VITAL_COLS + LAB_COLS:
        series = window_df[col].ffill().bfill().fillna(0.0)
        mean_val = float(series.mean())
        min_val  = float(series.min())
        max_val  = float(series.max())
        std_val  = float(series.std()) if len(series) > 1 else 0.0
        last_val = float(series.iloc[-1])
        features.extend([mean_val, min_val, max_val, std_val, last_val])

    # 3 demographic/admin features: Age, Gender, Max_ICULOS = 73 total
    age = float(window_df['Age'].iloc[0]) if 'Age' in window_df.columns else 60.0
    gender = float(window_df['Gender'].iloc[0]) if 'Gender' in window_df.columns else 0.0
    max_iculos = float(window_df['ICULOS'].max()) if 'ICULOS' in window_df.columns else 12.0
    features.extend([age, gender, max_iculos])

    return np.array(features, dtype=np.float32)

def extract_windows_and_features(df, target_pids, window_size=12, stride=6):
    """
    Extracts overlapping (7, 12) time-series windows & corresponding 73-D summaries.
    Target Leakage Fix: Evaluates deterioration at window termination (iloc[-1])
    to prevent post-onset vital signs from contaminating pre-onset prediction.
    """
    X_73d_list, X_ts_list, y_list = [], [], []

    for pid in target_pids:
        pdf = df[df['PatientID'] == pid].sort_values('Hour').reset_index(drop=True)
        num_rows = len(pdf)
        if num_rows < window_size:
            continue

        for start_idx in range(0, num_rows - window_size + 1, stride):
            window = pdf.iloc[start_idx : start_idx + window_size]

            # Extract 73-D summary vector
            f73 = compute_73d_summary(window)

            # Extract (7, 12) time-series tensor for 7 vital signs
            vitals_mat = window[VITAL_COLS].ffill().bfill().fillna(0.0).values.T  # Shape: (7, 12)

            # Target label at window termination (predicting active or imminent deterioration)
            label = int(window['SepsisLabel'].iloc[-1]) if 'SepsisLabel' in window.columns else 0

            X_73d_list.append(f73)
            X_ts_list.append(vitals_mat)
            y_list.append(label)

    return np.array(X_73d_list), np.array(X_ts_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

X_train_73d_raw, X_train_ts_raw, y_train = extract_windows_and_features(raw_df, train_pids)
X_val_73d_raw, X_val_ts_raw, y_val     = extract_windows_and_features(raw_df, val_pids)

# Fit StandardScaler strictly on training 73-D features
scaler_73d = StandardScaler()
X_train_73d = scaler_73d.fit_transform(X_train_73d_raw)
X_val_73d   = scaler_73d.transform(X_val_73d_raw)

# Standardize vital sign channels across samples and timesteps strictly on training cohort
ts_mean = X_train_ts_raw.mean(axis=(0, 2), keepdims=True)
ts_std  = X_train_ts_raw.std(axis=(0, 2), keepdims=True) + 1e-6
X_train_ts = (X_train_ts_raw - ts_mean) / ts_std
X_val_ts   = (X_val_ts_raw - ts_mean) / ts_std

print(f"[+] Extracted Slices -> Train Windows: {X_train_73d.shape[0]} | Val Windows: {X_val_73d.shape[0]}")
print(f"[+] Time-Series Window Shape: {X_train_ts.shape[1:]} (7 channels x 12 hourly timesteps)")
print(f"[+] Sepsis Windows: Train={int(y_train.sum())}/{len(y_train)} ({y_train.mean():.2%}) | Val={int(y_val.sum())}/{len(y_val)} ({y_val.mean():.2%})")

In [ ]:
# ============================================================
# 4. DATASET & DATALOADER CONSTRUCTION
# ============================================================
class SepsisDataset(Dataset):
    def __init__(self, X_73d, X_ts, y):
        self.X_73d = torch.tensor(X_73d, dtype=torch.float32)
        self.X_ts = torch.tensor(X_ts, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_73d[idx], self.X_ts[idx], self.y[idx]

# Create PyTorch Dataset and DataLoader instances
train_dataset = SepsisDataset(X_train_73d, X_train_ts, y_train)
val_dataset   = SepsisDataset(X_val_73d, X_val_ts, y_val)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"[+] Train DataLoader: {len(train_loader)} batches | Val DataLoader: {len(val_loader)} batches (Batch Size: {BATCH_SIZE})")

In [ ]:
# ============================================================
# 5. MODEL ARCHITECTURES & UPSTREAM CHECKPOINT INTEGRATION
# ============================================================

# --- A. Baseline 73-D MLP (Module 1/2 Architecture) ---
class Baseline73DMLP(nn.Module):
    """73-D Tabular MLP baseline with regularized hidden layers."""
    def __init__(self, input_dim=73, hidden_dims=[128, 64, 32], dropout_rate=0.2):
        super(Baseline73DMLP, self).__init__()
        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout_rate))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x_73d, x_ts=None):
        return self.net(x_73d)


# --- B. Multi-Scale 1D CNN Temporal Encoder (docs/interfaces.md compliant) ---
class ClinicalCNN1D(nn.Module):
    """
    Multi-Scale 1D-CNN Clinical Feature Extractor.
    Adheres strictly to docs/interfaces.md:
    Input: (Batch_Size, 7, 12) -> Output: (Batch_Size, 32)
    Parallel branches: k=3 (acute), k=5 (sub-acute), k=7 (sustained drift).
    """
    def __init__(self, in_channels=7, out_embed_dim=32, pool_type='avg'):
        super(ClinicalCNN1D, self).__init__()
        self.pool_type = pool_type.lower()
        self.branch_k3 = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm1d(16),
            nn.GELU()
        )
        self.branch_k5 = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.GELU()
        )
        self.branch_k7 = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=7, padding=3),
            nn.BatchNorm1d(16),
            nn.GELU()
        )
        # 16 + 16 + 16 = 48 concatenated multi-scale channels
        if self.pool_type == 'max':
            self.pool = nn.AdaptiveMaxPool1d(1)
        elif self.pool_type == 'avg':
            self.pool = nn.AdaptiveAvgPool1d(1)
        else:
            raise ValueError(f"pool_type must be 'avg' or 'max', got {pool_type!r}")

        self.fc_embed = nn.Sequential(
            nn.Linear(48, out_embed_dim),
            nn.GELU()
        )

    def forward(self, x_ts):
        feat_k3 = self.branch_k3(x_ts)  # (B, 16, 12)
        feat_k5 = self.branch_k5(x_ts)  # (B, 16, 12)
        feat_k7 = self.branch_k7(x_ts)  # (B, 16, 12)
        multi_scale_feat = torch.cat([feat_k3, feat_k5, feat_k7], dim=1)  # (B, 48, 12)
        pooled = self.pool(multi_scale_feat).squeeze(-1)                  # (B, 48)
        embedding = self.fc_embed(pooled)                                 # (B, 32)
        return embedding

# Interface Alias
MultiScale1DCNN = ClinicalCNN1D


# --- C. CNN-Only Classification Model ---
class CNNOnlyModel(nn.Module):
    def __init__(self, in_channels=7, out_embed_dim=32):
        super(CNNOnlyModel, self).__init__()
        self.cnn_encoder = ClinicalCNN1D(in_channels=in_channels, out_embed_dim=out_embed_dim)
        self.classifier = nn.Linear(out_embed_dim, 1)

    def forward(self, x_73d, x_ts):
        embed = self.cnn_encoder(x_ts)
        return self.classifier(embed)


# --- D. Module 3 SentryICU Fusion Model (73-D + 32-D CNN = 105-D) ---
class SentryICUFusionModel(nn.Module):
    """
    Multimodal Early Deterioration Fusion Model:
    Concatenates 73-D static summary features with 32-D temporal CNN embedding = 105-D.
    """
    def __init__(self, static_dim=73, ts_channels=7, cnn_embed_dim=32,
                 hidden_dims=[64, 32], dropout_rate=0.2, pool_type='avg'):
        super(SentryICUFusionModel, self).__init__()
        self.cnn_encoder = ClinicalCNN1D(in_channels=ts_channels, out_embed_dim=cnn_embed_dim, pool_type=pool_type)
        fusion_dim = static_dim + cnn_embed_dim  # 73 + 32 = 105

        layers = []
        in_dim = fusion_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout_rate))
            in_dim = h_dim

        layers.append(nn.Linear(in_dim, 1))
        self.fusion_mlp = nn.Sequential(*layers)

    def forward(self, x_73d, x_ts):
        cnn_embed = self.cnn_encoder(x_ts)               # (B, 32)
        fused = torch.cat([x_73d, cnn_embed], dim=1)      # (B, 105)
        logits = self.fusion_mlp(fused)                   # (B, 1)
        return logits, cnn_embed, fused

# Upstream Module 1 & Module 2 Checkpoint Discovery
m1_ckpt_path = os.path.join(PROJECT_DIR, 'models', 'module1_best_mlp.pt')
m2_ckpt_path = os.path.join(PROJECT_DIR, 'models', 'module2_best_regularized_mlp.pt')

if os.path.exists(m2_ckpt_path):
    print(f"[+] Found Upstream Module 2 Checkpoint: {m2_ckpt_path}")
elif os.path.exists(m1_ckpt_path):
    print(f"[+] Found Upstream Module 1 Checkpoint: {m1_ckpt_path}")
else:
    print(f"[*] Upstream checkpoints not found in {os.path.join(PROJECT_DIR, 'models')}  initializing fresh baseline.")

In [ ]:
# ============================================================
# 6. TRAINING & COMPREHENSIVE CLINICAL EVALUATION PIPELINE
# ============================================================

def compute_clinical_metrics(y_true, y_prob):
    """
    Computes comprehensive clinical metrics matching docs/interfaces.md:
    AUROC, AUPRC, Brier Score, ECE, Sensitivity, Specificity, F1.
    """
    y_true = np.array(y_true).flatten()
    y_prob = np.clip(np.array(y_prob).flatten(), 1e-7, 1.0 - 1e-7)

    # Area Under Curves
    auroc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5
    auprc = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.0

    # Calibration & Probability Error
    brier = brier_score_loss(y_true, y_prob)

    # Expected Calibration Error (ECE, 10 bins)
    bin_edges = np.linspace(0, 1, 11)
    ece = 0.0
    for i in range(10):
        in_bin = (y_prob >= bin_edges[i]) & (y_prob < bin_edges[i+1])
        prop_in_bin = in_bin.mean()
        if prop_in_bin > 0:
            avg_prob = y_prob[in_bin].mean()
            emp_acc = y_true[in_bin].mean()
            ece += np.abs(emp_acc - avg_prob) * prop_in_bin

    # Decision Metrics at Standard 0.5 Threshold
    preds_05 = (y_prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds_05, labels=[0, 1]).ravel()
    sensitivity = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    prec = precision_score(y_true, preds_05, zero_division=0)
    rec  = recall_score(y_true, preds_05, zero_division=0)
    f1   = f1_score(y_true, preds_05, zero_division=0)
    acc  = accuracy_score(y_true, preds_05)

    return {
        'ROC-AUC': float(auroc),
        'PR-AUC': float(auprc),
        'Brier Score': float(brier),
        'ECE': float(ece),
        'F1': float(f1),
        'Precision': float(prec),
        'Recall (Sens)': float(sensitivity),
        'Specificity': float(specificity),
        'Accuracy': float(acc)
    }

def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3, pos_weight=None):
    """
    Encapsulated training loop. Computes pos_weight directly from train_loader
    without global variable leakage.
    """
    model = model.to(DEVICE)

    # Clean class imbalance weighting from train_loader labels
    if pos_weight is None:
        all_labels = []
        for _, _, batch_y in train_loader:
            all_labels.extend(batch_y.numpy().flatten())
        all_labels = np.array(all_labels)
        pos_cnt = max(int((all_labels == 1).sum()), 1)
        neg_cnt = int((all_labels == 0).sum())
        pos_weight_val = neg_cnt / pos_cnt
    else:
        pos_weight_val = float(pos_weight)

    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_val], device=DEVICE))
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    for epoch in range(epochs):
        model.train()
        for x73, xts, y in train_loader:
            x73, xts, y = x73.to(DEVICE), xts.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x73, xts)
            logits = out[0] if isinstance(out, tuple) else out
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

    # Evaluation Phase
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for x73, xts, y in val_loader:
            x73, xts, y = x73.to(DEVICE), xts.to(DEVICE), y.to(DEVICE)
            out = model(x73, xts)
            logits = out[0] if isinstance(out, tuple) else out
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            val_preds.extend(probs)
            val_targets.extend(y.cpu().numpy().flatten())

    return compute_clinical_metrics(val_targets, val_preds)

In [ ]:
# ============================================================
# 7. EXPLICIT DIMENSION VERIFICATION, BASELINES & EXPORT
# ============================================================
print("\n" + "="*70)
print("FEATURE & FUSION REPRESENTATION SHAPE VERIFICATION")
print("="*70)

sample_73d, sample_ts, _ = next(iter(val_loader))
sample_73d, sample_ts = sample_73d.to(DEVICE), sample_ts.to(DEVICE)

temp_fusion_model = SentryICUFusionModel().to(DEVICE)
temp_fusion_model.eval()
with torch.no_grad():
    _, sample_cnn_embed, sample_fused = temp_fusion_model(sample_73d, sample_ts)

print(f"73-D static features:       {list(sample_73d.shape)} (docs/interfaces.md contract)")
print(f"CNN embedding:             {list(sample_cnn_embed.shape)} (32-D multi-scale representation)")
print(f"Fusion representation:     {list(sample_fused.shape)} (73-D + 32-D = 105-D)")
print("="*70 + "\n")

print("="*70)
print("TRAINING & EVALUATING COMPARATIVE BASELINES")
print("="*70)

set_seed(RANDOM_SEED)
model_a = Baseline73DMLP()
metrics_a = train_model(model_a, train_loader, val_loader, epochs=EPOCHS)

set_seed(RANDOM_SEED)
model_b = CNNOnlyModel()
metrics_b = train_model(model_b, train_loader, val_loader, epochs=EPOCHS)

set_seed(RANDOM_SEED)
model_c = SentryICUFusionModel()
metrics_c = train_model(model_c, train_loader, val_loader, epochs=EPOCHS)

baseline_results = pd.DataFrame([
    {'Model Architecture': 'A. Baseline 73-D MLP', **metrics_a},
    {'Model Architecture': 'B. CNN Only (32-D Embedding)', **metrics_b},
    {'Model Architecture': 'C. 73-D + CNN 32-D Fusion', **metrics_c}
])

print(baseline_results[['Model Architecture', 'ROC-AUC', 'PR-AUC', 'Brier Score', 'F1', 'Recall (Sens)', 'Specificity']].to_string(index=False))
print("="*70 + "\n")

# Checkpoint Export for Downstream Integration (Module 4 Sequence Model & Module 5 VAE)
export_dir = os.path.join(PROJECT_DIR, 'models')
os.makedirs(export_dir, exist_ok=True)
ckpt_save_path = os.path.join(export_dir, 'module3_best_cnn_fusion.pt')

torch.save({
    'model_state_dict': model_c.state_dict(),
    'cnn_state_dict': model_c.cnn_encoder.state_dict(),
    'ts_mean': ts_mean,
    'ts_std': ts_std,
    'scaler_73d_mean': scaler_73d.mean_,
    'scaler_73d_scale': scaler_73d.scale_,
    'vital_cols': VITAL_COLS,
    'lab_cols': LAB_COLS,
    'metrics': metrics_c,
    'fusion_dim': 105
}, ckpt_save_path)

print(f"[+] Successfully exported Module 3 checkpoint to: {ckpt_save_path}")

In [ ]:
# ============================================================
# 8. FAIR, NON-CONFOUNDED MODULE 3 ABLATION EXPERIMENTS
# ============================================================
if RUN_ABLATIONS:
    print("="*70)
    print("EXECUTING FAIR MODULE 3 ABLATION SUITE")
    print("="*70)

    ABLATION_EPOCHS = EPOCHS
    ablation_records = []

    # Standardized Fusion Head across all ablations to isolate convolutional inductive bias
    class FairAblationFusionModel(nn.Module):
        def __init__(self, encoder_module, embed_dim=32):
            super().__init__()
            self.encoder = encoder_module
            self.head = nn.Sequential(
                nn.Linear(73 + embed_dim, 64),
                nn.BatchNorm1d(64),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(64, 32),
                nn.BatchNorm1d(32),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(32, 1)
            )

        def forward(self, x73, xts):
            emb = self.encoder(xts)
            fused = torch.cat([x73, emb], dim=1)
            return self.head(fused)

    class SingleScaleEncoder(nn.Module):
        def __init__(self, k, pool_type='avg'):
            super().__init__()
            self.conv = nn.Sequential(
                nn.Conv1d(7, 32, kernel_size=k, padding=k // 2),
                nn.BatchNorm1d(32),
                nn.GELU()
            )
            self.pool = nn.AdaptiveMaxPool1d(1) if pool_type == 'max' else nn.AdaptiveAvgPool1d(1)

        def forward(self, xts):
            return self.pool(self.conv(xts)).squeeze(-1)

    class DeepCNNEncoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(7, 16, kernel_size=3, padding=1),
                nn.BatchNorm1d(16),
                nn.GELU(),
                nn.Conv1d(16, 32, kernel_size=3, padding=1),
                nn.BatchNorm1d(32),
                nn.GELU(),
                nn.AdaptiveAvgPool1d(1)
            )

        def forward(self, xts):
            return self.net(xts).squeeze(-1)

    # 1. Single-scale vs Multi-scale CNN
    set_seed(RANDOM_SEED)
    m_single = FairAblationFusionModel(SingleScaleEncoder(k=5), embed_dim=32)
    rec_single = train_model(m_single, train_loader, val_loader, epochs=ABLATION_EPOCHS)
    ablation_records.append({'Ablation Study': '1. Single-scale vs Multi-scale', 'Config': 'Single-Scale (k=5)', **rec_single})

    set_seed(RANDOM_SEED)
    m_multi = SentryICUFusionModel()
    rec_multi = train_model(m_multi, train_loader, val_loader, epochs=ABLATION_EPOCHS)
    ablation_records.append({'Ablation Study': '1. Single-scale vs Multi-scale', 'Config': 'Multi-Scale (k=3,5,7)', **rec_multi})

    # 2. Kernel-size comparison (k=3 vs 5 vs 7)
    for k_val in [3, 5, 7]:
        set_seed(RANDOM_SEED)
        m_k = FairAblationFusionModel(SingleScaleEncoder(k=k_val), embed_dim=32)
        rec_k = train_model(m_k, train_loader, val_loader, epochs=ABLATION_EPOCHS)
        ablation_records.append({'Ablation Study': '2. Kernel-size comparison', 'Config': f'Kernel Size k={k_val}', **rec_k})

    # 3. Pooling comparison (AvgPool vs MaxPool)
    set_seed(RANDOM_SEED)
    m_max = SentryICUFusionModel(pool_type='max')
    rec_max = train_model(m_max, train_loader, val_loader, epochs=ABLATION_EPOCHS)
    ablation_records.append({'Ablation Study': '3. Pooling comparison', 'Config': 'Adaptive Max Pooling', **rec_max})

    set_seed(RANDOM_SEED)
    m_avg = SentryICUFusionModel(pool_type='avg')
    rec_avg = train_model(m_avg, train_loader, val_loader, epochs=ABLATION_EPOCHS)
    ablation_records.append({'Ablation Study': '3. Pooling comparison', 'Config': 'Adaptive Avg Pooling', **rec_avg})

    # 4. Network depth comparison (1-layer vs 2-layer CNN)
    set_seed(RANDOM_SEED)
    m_deep = FairAblationFusionModel(DeepCNNEncoder(), embed_dim=32)
    rec_deep = train_model(m_deep, train_loader, val_loader, epochs=ABLATION_EPOCHS)
    ablation_records.append({'Ablation Study': '4. Network depth comparison', 'Config': '2-Layer Conv Depth', **rec_deep})

    # 5. Component comparison
    set_seed(RANDOM_SEED)
    m_mlp_only = Baseline73DMLP()
    rec_mlp_only = train_model(m_mlp_only, train_loader, val_loader, epochs=ABLATION_EPOCHS)

    set_seed(RANDOM_SEED)
    m_cnn_only = CNNOnlyModel()
    rec_cnn_only = train_model(m_cnn_only, train_loader, val_loader, epochs=ABLATION_EPOCHS)

    set_seed(RANDOM_SEED)
    m_full_fusion = SentryICUFusionModel()
    rec_full_fusion = train_model(m_full_fusion, train_loader, val_loader, epochs=ABLATION_EPOCHS)

    ablation_records.append({'Ablation Study': '5. CNN vs MLP', 'Config': 'CNN Only (32-D)', **rec_cnn_only})
    ablation_records.append({'Ablation Study': '5. CNN vs MLP', 'Config': 'MLP Only (73-D)', **rec_mlp_only})
    ablation_records.append({'Ablation Study': '6. Fusion vs MLP Alone', 'Config': 'Full 105-D Fusion', **rec_full_fusion})
    ablation_records.append({'Ablation Study': '6. Fusion vs MLP Alone', 'Config': 'MLP Only', **rec_mlp_only})

    df_ablations = pd.DataFrame(ablation_records)
    print(df_ablations[['Ablation Study', 'Config', 'ROC-AUC', 'PR-AUC', 'Brier Score', 'F1', 'Recall (Sens)', 'Specificity']].to_string(index=False))
    print("="*70 + "\n")

In [ ]:
# ============================================================
# 9. 1D TEMPORAL GRAD-CAM / PHYSIOLOGICAL ATTRIBUTION
# ============================================================
print("="*70)
print("GENERATING 1D TEMPORAL PHYSIOLOGICAL ATTRIBUTION HEATMAP")
print("="*70)

# Search for a positive-label sample in the validation split
sample_x73, sample_xts, sample_y = None, None, None
for batch_x73, batch_xts, batch_y in val_loader:
    pos_idx = (batch_y.squeeze(-1) == 1).nonzero(as_tuple=True)[0]
    if len(pos_idx) > 0:
        idx = pos_idx[0].item()
        sample_x73 = batch_x73[idx:idx + 1].to(DEVICE)
        sample_xts = batch_xts[idx:idx + 1].to(DEVICE)
        sample_y = batch_y[idx:idx + 1]
        break

if sample_x73 is None:
    sample_x73, sample_xts, sample_y = next(iter(val_loader))
    sample_x73 = sample_x73[0:1].to(DEVICE)
    sample_xts = sample_xts[0:1].to(DEVICE)
    sample_y = sample_y[0:1]

label_val = int(sample_y.flatten()[0].item())
print(f"[+] Sample Diagnosis Label: {label_val} ({'High-Risk / Sepsis' if label_val == 1 else 'Low-Risk / Control'})")

# Enable gradient tracking on input vital signs
sample_xts = sample_xts.clone().detach().requires_grad_(True)
model_c.eval()
logits, _, _ = model_c(sample_x73, sample_xts)
score = logits[0, 0]
model_c.zero_grad()
score.backward()

# Attribution: Magnitude-aware Gradient x Activation
# Captures acute vital elevations (tachycardia) AND acute depressions (hypotension, hypothermia)
grads = sample_xts.grad.data.cpu().numpy()[0]            # Shape: (7, 12)
vitals_norm = sample_xts.data.cpu().numpy()[0]           # Shape: (7, 12)

# Absolute gradient-weighted saliency (prevents negative vital sign blinding)
attribution = np.abs(vitals_norm * grads)
attr_range = attribution.max() - attribution.min()
if attr_range > 1e-6:
    attribution = (attribution - attribution.min()) / attr_range
else:
    attribution = np.zeros_like(attribution)

# Visualization
plt.figure(figsize=(11, 5), dpi=120)
sns.heatmap(
    attribution,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    xticklabels=[f"t={h}h" for h in range(1, 13)],
    yticklabels=VITAL_COLS,
    cbar_kws={'label': 'Normalized Attribution Score'}
)
plt.title("Module 3: 1D Temporal Physiological Attribution (Vital Signs x 12-Hour Window)", fontsize=12, fontweight='bold')
plt.xlabel("Observation Timeline (Hourly Windows)", fontsize=11)
plt.ylabel("Monitored ICU Vital Signs", fontsize=11)
plt.tight_layout()
plt.show()

print("[+] Module 3 Execution Completed Successfully!")